In [180]:
from abc import ABC, abstractmethod
from dataclasses import dataclass
from typing import Annotated

import numpy as np

from geneticengine.grammar.metahandlers.ints import IntRange
from geneticengine.grammar import extract_grammar
from geneticengine.problems import SingleObjectiveProblem, MultiObjectiveProblem
from geneticengine.random.sources import NativeRandomSource
from geneticengine.algorithms.gp.gp import GeneticProgramming
from geneticengine.evaluation.budget import TimeBudget, EvaluationBudget
from geneticengine.representations.tree.initializations import MaxDepthDecider
from geneticengine.representations.tree.treebased import TreeBasedRepresentation
from geneticengine.evaluation.recorder import CSVSearchRecorder
from geneticengine.evaluation.tracker import ProgressTracker

from sklearn.datasets import load_breast_cancer

import pandas as pd

from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import f1_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

import time

In [181]:
df = pd.read_csv('creditcard.csv') #https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud
df.drop(columns=['Time'], inplace=True)
fraction = 0.1
n_rows_keep = int(len(df) * fraction)
df = df.head(n_rows_keep).copy()

df.info()

# df = load_breast_cancer(as_frame=True).frame
# df.head(3)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 28480 entries, 0 to 28479
Data columns (total 30 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   V1      28480 non-null  float64
 1   V2      28480 non-null  float64
 2   V3      28480 non-null  float64
 3   V4      28480 non-null  float64
 4   V5      28480 non-null  float64
 5   V6      28480 non-null  float64
 6   V7      28480 non-null  float64
 7   V8      28480 non-null  float64
 8   V9      28480 non-null  float64
 9   V10     28480 non-null  float64
 10  V11     28480 non-null  float64
 11  V12     28480 non-null  float64
 12  V13     28480 non-null  float64
 13  V14     28480 non-null  float64
 14  V15     28480 non-null  float64
 15  V16     28480 non-null  float64
 16  V17     28480 non-null  float64
 17  V18     28480 non-null  float64
 18  V19     28480 non-null  float64
 19  V20     28480 non-null  float64
 20  V21     28480 non-null  float64
 21  V22     28480 non-null  float64
 22

In [182]:
target_var = 'Class'

split_percentage = 0.8
split_point = int(len(df) * split_percentage)

train_df = df.iloc[:split_point]
test_df = df.iloc[split_point:]

X_train = train_df.drop(target_var, axis=1)
y_train = train_df[target_var]
X_test = test_df.drop(target_var, axis=1)
y_test = test_df[target_var]

In [183]:
scaler = StandardScaler()
X_train['Amount'] = scaler.fit_transform(X_train[['Amount']])
X_test['Amount'] = scaler.transform(X_test[['Amount']])

In [184]:
feature_names = X_train.columns.tolist()
n_features = len(feature_names)

In [185]:
baseline_model = DecisionTreeClassifier(random_state=42)

baseline_model.fit(X_train, y_train)

baseline_y_pred = baseline_model.predict(X_test)

baseline_f1 = f1_score(y_test, baseline_y_pred)

print(baseline_f1)

0.8571428571428571


In [186]:
class Value(ABC):
    def evaluate(self):
        pass

class Scalar(ABC):
    pass

class Vectorial(ABC):
    pass

In [187]:
@dataclass
class PastValues(Vectorial):
    index: Annotated[int, IntRange(0, n_features-1)]
    lockback: Annotated[int, IntRange(5, 10)]

    def evaluate(self, X, i):
        start_index = max(0, i-self.lockback+1)
        values = X.iloc[start_index: i+1, self.index].values
        if len(values) < self.lockback: #if there are not enough past values
            padding = np.zeros(self.lockback - len(values))
            values = np.concatenate((padding, values))
        return values
    def __str__(self):
        return f"past_values({feature_names[self.index]}, {self.lockback})"

In [188]:
@dataclass
class Mean(Scalar):
    arr: Vectorial
    def evaluate(self, X, i):
        v = self.arr.evaluate(X,i)
        return np.mean(v)
    def __str__(self):
        return f"mean({self.arr})"
    
@dataclass
class Max(Scalar):
    arr: Vectorial
    def evaluate(self, X, i):
        v = self.arr.evaluate(X,i)
        return np.max(v)
    def __str__(self):
        return f"max({self.arr})"
    
@dataclass
class Min(Scalar):
    arr: Vectorial
    def evaluate(self, X, i):
        v = self.arr.evaluate(X,i)
        return np.min(v)
    def __str__(self):
        return f"min({self.arr})"


In [189]:
@dataclass #Scalar Features (1)
class ScalarVar(Scalar): 
    index: Annotated[int, IntRange(0,n_features-1)]

    def evaluate(self, X, i):
        return X.iloc[i, self.index]
    
    def __str__(self):
        return feature_names[self.index]

# @dataclass #Vectorial Features ([1,2,3,4])
# class VectorialVar(Vectorial):
#     index: Annotated[int, IntRange(1,2)]
    
#     def evaluate(self, X):
#         return X[self.index]

In [190]:
#scalar -> scalar
@dataclass 
class Add(Scalar):
    right: Scalar
    left: Scalar

    def evaluate(self, X, i):
        return self.left.evaluate(X,i) + self.right.evaluate(X,i)
    
    def __str__(self):
        return f"({self.left} + {self.right})"

@dataclass
class Subtract(Scalar):
    right: Scalar
    left: Scalar

    def evaluate(self, X, i):
        return float(self.left.evaluate(X,i)) - float(self.right.evaluate(X,i))
    
    def __str__(self):
        return f"({self.left} - {self.right})"
    
@dataclass
class Multiply(Scalar):
    right: Scalar
    left: Scalar

    def evaluate(self, X, i):
        return self.left.evaluate(X,i) * self.right.evaluate(X,i)
    
    def __str__(self):
        return f"({self.left} * {self.right})"

In [191]:
grammar = extract_grammar([PastValues, Mean, Max, Min, ScalarVar, Add, Subtract, Multiply], Scalar)
print(f"Grammar: {repr(grammar)}")

Grammar: Grammar<Starting=Scalar,Productions={
Scalar -> Mean(arr: Vectorial)|
	Max(arr: Vectorial)|
	Min(arr: Vectorial)|
	ScalarVar(index: Annotated[int])|
	Add(right: Scalar, left: Scalar)|
	Subtract(right: Scalar, left: Scalar)|
	Multiply(right: Scalar, left: Scalar)

Vectorial -> PastValues(index: Annotated[int], lockback: Annotated[int])
}


In [192]:
def fitness_function(individual: Value):    
    start = time.perf_counter()

    train_feature = [individual.evaluate(X_train, i) for i in range(len(X_train))]
    test_feature = [individual.evaluate(X_test, i) for i in range(len(X_test))]
    
    X_train_new = np.array(train_feature).reshape(-1,1)
    X_test_new = np.array(test_feature).reshape(-1,1)

    model = DecisionTreeClassifier(random_state=42)
    model.fit(X_train_new, y_train)
    y_pred = model.predict(X_test_new)

    elapsed = time.perf_counter() - start

    return float(elapsed), f1_score(y_test, y_pred)

In [193]:
prob = MultiObjectiveProblem(
    fitness_function=fitness_function,
    minimize=[True, False],
)
r = NativeRandomSource(123)
alg = GeneticProgramming(
    problem=prob,
    budget=TimeBudget(600),
    population_size=20,
    representation=TreeBasedRepresentation(grammar, MaxDepthDecider(r, grammar, 5)),
    random=r,
    tracker=ProgressTracker(
        prob,
        recorders=[CSVSearchRecorder(
            csv_path='outpupt.csv', 
            problem=prob, 
            fields={"Eval Time": lambda t,i,p: i.get_fitness(p).fitness_components[0],
                    "F1 Score": lambda t,i,p: i.get_fitness(p).fitness_components[1],
                    "Expression": lambda t, i, p: i.get_phenotype(),
                    },
            only_record_best_individuals=False)]
    )
    
)

solutions = alg.search()

In [194]:
best_sol = max(solutions, key=lambda row: row.get_fitness(prob).fitness_components[1])
print(best_sol.get_phenotype(), best_sol.get_fitness(prob))

(min(past_values(V11, 7)) * ((V22 + mean(past_values(V24, 5))) - ((V12 + V17) + V22))) [4.865326999977697,0.3157894736842105]


In [195]:
# test = Max(arr=PastValues(index=feature_names.index('worst fractal dimension'), lockback=4))

# generated_value = [test.evaluate(X_test, i) for i in range(len(X_test))]

# test_df = X_test.copy()
# test_df['generated'] = generated_value
# test_df['target_x'] = y_test
# test_df.head(10)